In [7]:
from chore_tracker.routes import routes_bp
from flask import Flask, render_template, request, redirect, session, url_for
from chore_tracker.utils import Config,get_db_connection, ChoreActions, ChoreData, UserActions
from datetime import date

conn = get_db_connection()

In [2]:
c = conn.execute('''SELECT * FROM completed_chores''').fetchall()
[dict(row) for row in c]

[]

In [3]:
action = ChoreActions(conn)

Child ID: 2, Earned: 0, Behavior Deductions: 0, Expenses: 0, Net: 0
Child ID: 3, Earned: 0, Behavior Deductions: 0, Expenses: 0, Net: 0
Child ID: 1, Earned: 0, Behavior Deductions: 0, Expenses: 0, Net: 0


In [4]:
users = UserActions(conn)

In [5]:
users.fetch_user_id(username='Evelyn')

2

In [6]:
data = ChoreData(conn)

Child ID: 2, Earned: 0, Behavior Deductions: 0, Expenses: 0, Net: 0
Child ID: 3, Earned: 0, Behavior Deductions: 0, Expenses: 0, Net: 0
Child ID: 1, Earned: 0, Behavior Deductions: 0, Expenses: 0, Net: 0


In [9]:
from datetime import datetime, timedelta
today = datetime.now().date()  # Get today's date without time
thirty_days_ago = today - timedelta(days=30)  # 30 days ago from today 
completed_chores = conn.execute('''
            SELECT u.name as child_name, c.completion_date, COUNT(*) as count
            FROM completed_chores c
            JOIN users u ON c.user_id = u.id
            WHERE c.completion_date BETWEEN ? AND ?
            GROUP BY u.name, c.completion_date
            ORDER BY u.name, c.completion_date
        ''', (thirty_days_ago, today)).fetchall()  # Note the order: (start_date, end_date)
completed_chores

/tmp/ipykernel_108740/3097284128.py:4: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  completed_chores = conn.execute('''


[]

In [10]:
data.completed_chores_timeline()


'{}'

In [7]:
quick_id = action.fetch_choreid('10 Minute Helpfulness','preset','Any')
completion_date = date.today().isoformat()
action.complete_chore(2,quick_id,completion_date)

TypeError: 'NoneType' object is not subscriptable

In [8]:
children = data.fetch_children()
earnings = data.get_earnings_report()
print(earnings)

Child ID: 2, Earned: 0, Behavior Deductions: 0, Expenses: 0, Net: 0
Child ID: 3, Earned: 0, Behavior Deductions: 0, Expenses: 0, Net: 0
Child ID: 1, Earned: 0, Behavior Deductions: 0, Expenses: 0, Net: 0
[{'name': 'Evelyn', 'net_earnings': 0}, {'name': 'Lucy', 'net_earnings': 0}, {'name': 'Virginia', 'net_earnings': 0}]


In [11]:

def calculate_earnings(minutes):
    if minutes < 0:
        return min(float((minutes/60)* cfg.hourly_rate), float(cfg.minimum_deduct))
    else:
        return max((minutes / 60) * cfg.hourly_rate, cfg.minimum_rate)

In [15]:

cfg = Config.from_yaml()
min(float(calculate_earnings(-5)),float(cfg.minimum_deduct))

-1.0

In [16]:
print([dict(x) for x in children])

[{'id': 2, 'name': 'Virginia'}, {'id': 3, 'name': 'Evelyn'}, {'id': 4, 'name': 'Lucy'}]


In [17]:
quick_id = action.fetch_choreid('10 Minute Helpfulness','preset','Any')
completion_date = date.today().isoformat()
action.complete_chore(2,quick_id,completion_date)

3,2,1.0,2025-02-20
3,2,1.0,2025-02-20


In [18]:
action.fetch_chores()

In [19]:
dict(action.morning_chores[0])

{'id': 7, 'name': 'Brush Teeth', 'preset_amount': 1.0}

In [9]:
chores = [dict(x) for x in action.conn.execute('SELECT * FROM chores').fetchall()]
chores[:5]

[{'id': 1,
  'name': '20 minute reading',
  'assigned': 'Evelyn',
  'time': 20.0,
  'type': 'school'},
 {'id': 2,
  'name': '20 minute reading',
  'assigned': 'Virginia',
  'time': 20.0,
  'type': 'school'},
 {'id': 3,
  'name': '10 minute reading',
  'assigned': 'Lucy',
  'time': 10.0,
  'type': 'school'}]

In [11]:
action.conn.execute('SELECT time FROM chores WHERE id = ?', (1,)).fetchone()['time']

20.0

In [15]:
def calculate_minutes_for_earnings(earnings):
    return round((earnings * 60) / cfg.hourly_rate)

In [14]:
action.calculate_minutes_for_earnings(52)

260

In [15]:
data.chores

In [16]:
data.behavior_deductions

[{'name': 'Evelyn', 'behavior_deductions': 0},
 {'name': 'Lucy', 'behavior_deductions': 0},
 {'name': 'Virginia', 'behavior_deductions': 0}]

In [17]:
data = ChoreData(conn)
chores = data.fetch_all_chores()
[chore['name'] for chore in chores]

Child ID: 2, Earned: 0, Behavior Deductions: 0, Expenses: 0, Net: 0
Child ID: 3, Earned: 0, Behavior Deductions: 0, Expenses: 0, Net: 0
Child ID: 1, Earned: 0, Behavior Deductions: 0, Expenses: 0, Net: 0


['20 minute reading', '20 minute reading', '10 minute reading']

In [8]:
users = UserActions(conn)
users_names = users.fetch_user_names()
users_names = [dict(x)['name'] for x in users_names]
users_names


['Evelyn', 'Lucy', 'Virginia']

In [15]:
from datetime import datetime, timedelta
import random
def add_sample_data(conn,users,num =10):
    data = ChoreData(conn)
    actions = ChoreActions(conn)
    today = datetime.now().date()  # Get today's date without time
    thirty_days_ago = today - timedelta(days=30)  # 30 days ago from today
    
    chores =  data.fetch_all_chores()
    for user in users:
        # add user,type,and password to users table
        conn.execute('INSERT OR IGNORE INTO users (name, role, password) VALUES (?, ?, ?)',
                     (user, 'child', 'password'))
        # add 10 random completed chores for each user
        # select 10 random dates from last 30 days
        for date in range(num):
            # select random date from last 30 days
            random_date = today - timedelta(days=random.randint(0, 30))
            # select random chore from chores
            random_chore = random.choice(chores)
            # select random amount from 1 to 10
            random_amount = random.randint(1, num)
            earnings = actions.calculate_earnings(random_amount)
            # fetch user id from users
            user_id = conn.execute('SELECT id FROM users WHERE name = ?', (user,)).fetchone()['id']
            # fetch chore id from chores
            chore_id = conn.execute('SELECT id FROM chores WHERE name = ?', (random_chore['name'],)).fetchone()['id']
            # insert into completed_chores
            conn.execute('INSERT INTO completed_chores (user_id, chore_id, amount_earned, time, completion_date) VALUES (?, ?, ?, ?, ?)',
                         (user_id, chore_id, earnings, random_amount, random_date,))
       
    # commit changes
    conn.commit()

def remove_sample_data(conn):
    users ['Bob', 'Alice', 'Charlie', 'Diana']
    # delete all data from completed_chores from listed users:
    for user in users:
        conn.execute('DELETE FROM completed_chores WHERE user_id = (SELECT id FROM users WHERE name = ?)', (user,))
    # delete all data from chores from listed users
    for user in users:
        conn.execute('DELETE FROM users WHERE name = ?', (user,))
    # delete all data from users
    conn.commit() 

In [18]:
add_sample_data(conn,users_names,10)

Child ID: 2, Earned: 21.0, Behavior Deductions: 0, Expenses: 0, Net: 21.0
Child ID: 3, Earned: 23.75, Behavior Deductions: 0, Expenses: 0, Net: 23.75
Child ID: 1, Earned: 21.0, Behavior Deductions: 0, Expenses: 0, Net: 21.0
Child ID: 2, Earned: 21.0, Behavior Deductions: 0, Expenses: 0, Net: 21.0
Child ID: 3, Earned: 23.75, Behavior Deductions: 0, Expenses: 0, Net: 23.75
Child ID: 1, Earned: 21.0, Behavior Deductions: 0, Expenses: 0, Net: 21.0


/tmp/ipykernel_113149/283873935.py:29: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  conn.execute('INSERT INTO completed_chores (user_id, chore_id, amount_earned, time, completion_date) VALUES (?, ?, ?, ?, ?)',


In [6]:
import sqlite3

def execute_sql_file(conn, sql_file_path):
    """
    Executes an SQL file against the SQLite database.

    Args:
        db_path (str): Path to the SQLite database file.
        sql_file_path (str): Path to the SQL file containing DROP TABLE statements.
    """
    try:
    
        
        # Read SQL file
        with open(sql_file_path, 'r') as sql_file:
            sql_script = sql_file.read()
        
        # Execute SQL script
        conn.executescript(sql_script)
        
        # Commit and close connection
        conn.commit()

        print("SQL script executed successfully.")

    except Exception as e:
        print(f"Error executing SQL file: {e}")

# Example usage
execute_sql_file(conn, "./chore_tracker/sql/reset_tables.sql")
execute_sql_file(conn, "./chore_tracker/sql/schema.sql")


SQL script executed successfully.
SQL script executed successfully.
